# Stage 2 follow-up: No-Show Breakdown
## ISM 6562 — Final Project — MediStream Telehealth
---
Closes a gap in `02-spark-transforms.ipynb`: the brief asks for no-show rates broken out by **specialty × time-of-day × day-of-week × visit_type** (Q1: No-Show Prediction). The original notebook produces an overall no-show rate per physician but no time-/day-bucketed breakdown.

**Input:** `/medistream/curated/appointments` (Parquet, written by `02-spark-transforms.ipynb`)

**Output:** `/medistream/analytics/no_show_breakdown` (Parquet, partitioned by `specialty`)

**Output schema:** `specialty, visit_type, day_of_week, time_of_day, total_appointments, no_show_count, no_show_rate_pct`

## Setup

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName('MediStream-Stage2b-NoShowBreakdown')
    .master('spark://spark-master:7077')
    .config('spark.executor.memory', '1g')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate())

print(f'Spark version: {spark.version}')
print(f'Application ID: {spark.sparkContext.applicationId}')

In [ ]:
# Resolve curated/analytics paths — try HDFS, fall back to local mount.
# Mirrors the path-resolution pattern in 02-spark-transforms.ipynb so this
# notebook runs in either environment without edits.
HDFS_CURATED   = 'hdfs://namenode:9000/medistream/curated'
HDFS_ANALYTICS = 'hdfs://namenode:9000/medistream/analytics'
LOCAL_CURATED   = '/home/jovyan/data/curated'
LOCAL_ANALYTICS = '/home/jovyan/data/analytics'

try:
    spark.range(1).write.mode('overwrite').parquet(f'{HDFS_ANALYTICS}/_probe')
    CURATED, ANALYTICS = HDFS_CURATED, HDFS_ANALYTICS
    print('Using HDFS')
except Exception:
    CURATED, ANALYTICS = LOCAL_CURATED, LOCAL_ANALYTICS
    print('HDFS unavailable, using local mount')

print(f'  curated:   {CURATED}')
print(f'  analytics: {ANALYTICS}')

## Derive time features

`scheduled_time` was cast to timestamp by `curate_dataset()` in the upstream notebook (any column containing `date` or `time` gets a `to_timestamp`). From it we derive:
- `hour_of_day` — 0..23
- `day_of_week` — "Monday".."Sunday" (`EEEE` format keeps it human-readable for analysts)
- `time_of_day` — categorical bucket: morning (06–12), afternoon (12–17), evening (17–22), night (22–06)
- `is_no_show` — binary flag where `status == 'no_show'`. The brief defines status values as `completed`, `cancelled`, `no_show`.

In [ ]:
appts = spark.read.parquet(f'{CURATED}/appointments')
print(f'Loaded {appts.count():,} appointment rows')
appts.printSchema()

In [ ]:
def time_of_day_bucket(hour_col):
    return (F.when((hour_col >= 6) & (hour_col < 12), 'morning')
             .when((hour_col >= 12) & (hour_col < 17), 'afternoon')
             .when((hour_col >= 17) & (hour_col < 22), 'evening')
             .otherwise('night'))

appts_t = (appts
    .withColumn('hour_of_day', F.hour('scheduled_time'))
    .withColumn('day_of_week', F.date_format('scheduled_time', 'EEEE'))
    .withColumn('time_of_day', time_of_day_bucket(F.col('hour_of_day')))
    .withColumn('is_no_show', F.when(F.lower(F.col('status')) == 'no_show', 1).otherwise(0))
)
appts_t.select('appointment_id', 'specialty', 'visit_type', 'scheduled_time',
               'hour_of_day', 'day_of_week', 'time_of_day', 'status', 'is_no_show').show(8, truncate=False)

## Aggregate no-show breakdown

Group by the four dimensions the brief calls out and compute counts + rate. Write Parquet **partitioned by `specialty`** so downstream queries that filter to one specialty (the common pattern for physician-facing dashboards) read only the relevant partition.

In [ ]:
breakdown = (appts_t
    .groupBy('specialty', 'visit_type', 'day_of_week', 'time_of_day')
    .agg(
        F.count('*').alias('total_appointments'),
        F.sum('is_no_show').alias('no_show_count'),
        F.round(F.avg('is_no_show') * 100, 2).alias('no_show_rate_pct'),
    )
    .orderBy(F.desc('no_show_rate_pct'))
)

out_path = f'{ANALYTICS}/no_show_breakdown'
(breakdown.write
    .mode('overwrite')
    .partitionBy('specialty')
    .parquet(out_path))

print(f'Wrote {breakdown.count():,} rows to {out_path}')

## Verify

In [ ]:
verify = spark.read.parquet(f'{ANALYTICS}/no_show_breakdown')
print(f'Read back {verify.count():,} rows, {len(verify.columns)} columns')
verify.printSchema()
print('\nTop 15 highest no-show buckets:')
verify.orderBy(F.desc('no_show_rate_pct')).show(15, truncate=False)
print('\nLowest 10 no-show buckets:')
verify.orderBy('no_show_rate_pct').show(10, truncate=False)